### Single Qubit Dataset Creation ans Saving

In [2]:
import h5py
from typing import Tuple, Optional
from loguru import logger
import numpy as np
import itertools
from typing import Union

In [3]:
import pickle as pkl

In [4]:
import numpy as np
import h5py
import os
import logging

def hdf5_data_save(data_path: str, data_file_name: str, X_data: np.ndarray, y_data: np.ndarray, data_type: str):
    """
    Save datasets to an HDF5 file based on the specified data type.

    Parameters:
        data_path (str): The path to the directory where the HDF5 file will be stored.
        data_file_name (str): The name of the HDF5 file.
        X_data (np.ndarray): The feature matrix to save.
        y_data (np.ndarray): The labels associated with the feature matrix.
        data_type (str): The type of dataset to save ('Train' or 'Test').

    Returns:
        bool: True if the data is saved successfully, False otherwise.
    """

    # Ensure the data path exists, if not, create it
    if not os.path.exists(data_path):
        os.makedirs(data_path, exist_ok=True)

    # Full path to the HDF5 file
    file_path = os.path.join(data_path, data_file_name)

    # Dictionary to map data types to dataset keys in the HDF5 file
    data_keys = {
        "Train": ("X_train", "y_train"),
        "Test": ("X_test", "y_test")
    }

    # Retrieve appropriate dataset keys based on specified data_type
    dataset_keys = data_keys.get(data_type)
    
    # If the data_type is not recognized, log error and return False
    if not dataset_keys:
        logging.error(f"Data type '{data_type}' is not supported. Choose 'Train' or 'Test'.")
        return False

    try:
        # Using a context manager to handle the file
        with h5py.File(file_path, 'a') as hf:  # 'a' mode to append or create new
            # Write datasets to the HDF5 file
            hf.create_dataset(dataset_keys[0], data=X_data)
            hf.create_dataset(dataset_keys[1], data=y_data)
            logging.info(f"Data successfully saved for '{data_type}' in {file_path}")
        return True
    except Exception as e:
        logging.error(f"Failed to save data: {str(e)}")
        return False


In [5]:
def hdf5_data_load(data_path: str, data_file_name: str, data_type: str, return_portion: int = 1) -> (
        Tuple)[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Load dataset from an HDF5 file based on the specified data type.

    Parameters:
        data_path (str): The path to the directory containing the HDF5 file.
        data_file_name (str): The name of the HDF5 file.
        data_type (str): The type of dataset to load ('Train' or 'Test').

    Returns:
        Tuple[Optional[np.ndarray], Optional[np.ndarray]]: A tuple containing the feature
        matrix (X) and labels (y), or (None, None) if an error occurs.
    """

    # Dictionary to map data types to dataset keys in the HDF5 file
    data_keys = {
        "Train": ("X_train", "y_train"),
        "Test": ("X_test", "y_test")
    }

    # Retrieve appropriate dataset keys based on specified data_type
    dataset_keys = data_keys.get(data_type)

    # If the data_type is not recognized, return None
    if not dataset_keys:
        logger.error(f"Data type '{data_type}' is not supported. Choose 'Train' or 'Test'.")
        return None, None

    # Using a context manager to handle the file
    with h5py.File(f"{data_path}/{data_file_name}", 'r') as hf:
        if return_portion > 1:
            X_dset = hf[dataset_keys[0]]
            y_dset = hf[dataset_keys[1]]
            num_samples_to_load = X_dset.shape[0] // return_portion

            X = np.array(X_dset[:num_samples_to_load])
            y = np.array(y_dset[:num_samples_to_load])
        else:
            X = np.array(hf[dataset_keys[0]])
            y = np.array(hf[dataset_keys[1]])
    return X, y


In [6]:
def get_dataset_labels(y,bit_only=False):

    q_map = {
        "Q5": set(),
        "Q4": set(),
        "Q3": set(),
        "Q2": set(),
        "Q1": set(),
    }

    for label in y:
        # print(label)
        encoding = np.unpackbits(np.array([label], dtype=np.uint8))[-5:][::-1]
    
        for i in range(len(encoding)):

            if bit_only:
                if encoding[i] == 1 and sum(encoding)==1:
                    q_map[f"Q{i+1}"].add(label)
            else:
                if encoding[i] == 1:
                    q_map[f"Q{i+1}"].add(label)

    return q_map

In [7]:
def saving_data(X, y, label_dict, dir, filename, data_type):
    for q_key, correct_labels in label_dict.items():
        qubit_file_name = filename + "_" + q_key
        
        y_binary = np.isin(y, list(correct_labels)).astype(int)
        hdf5_data_save(dir, qubit_file_name, X, y_binary, data_type)
        print(qubit_file_name, "created succesfully", "unique labels:", np.unique(y_binary), len(y_binary[y_binary==1]))
        y_binary = None
    

***

In [8]:
# data_train_file_name = 'DRaw_C_Tr_v0-001'
data_path = "./five_qubit_data"

In [9]:
single_qubit_dataset_path = "~/Storage/data"
train_file_name = "DRaw_C_Tr_v0-001"
test_file_name = "DRaw_C_Te_v0-002"

In [10]:
#X_train, y_train = hdf5_data_load(data_path, train_file_name, data_type='Train')


In [14]:
X_test, y_test = hdf5_data_load(data_path, test_file_name, data_type='Test', return_portion=3)

In [15]:
test_q_maps = get_dataset_labels(y_test)

In [ ]:
#train_q_maps = get_dataset_labels(y_train)

In [16]:
#train_q_maps
test_q_maps

{'Q5': set(),
 'Q4': {np.int32(8), np.int32(9), np.int32(10)},
 'Q3': {np.int32(4), np.int32(5), np.int32(6), np.int32(7)},
 'Q2': {np.int32(2), np.int32(3), np.int32(6), np.int32(7), np.int32(10)},
 'Q1': {np.int32(1), np.int32(3), np.int32(5), np.int32(7), np.int32(9)}}

In [17]:
#saving_data(X_train, y_train, train_q_maps, single_qubit_dataset_path, train_file_name, data_type = "Train" )


saving_data(X_test, y_test, test_q_maps, single_qubit_dataset_path, test_file_name, data_type = "Test" )

DRaw_C_Te_v0-002_Q5 created succesfully unique labels: [0] 0
DRaw_C_Te_v0-002_Q4 created succesfully unique labels: [0 1] 93333
DRaw_C_Te_v0-002_Q3 created succesfully unique labels: [0 1] 140000
DRaw_C_Te_v0-002_Q2 created succesfully unique labels: [0 1] 163333
DRaw_C_Te_v0-002_Q1 created succesfully unique labels: [0 1] 175000


In [29]:
del X_train, y_train

In [ ]:
X_test, y_test = hdf5_data_load(data_path, test_file_name, data_type='Test')

In [24]:
test_q_maps = get_dataset_labels(y_test)

In [25]:
test_q_maps

{'Q5': {16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31},
 'Q4': {8, 9, 10, 11, 12, 13, 14, 15, 24, 25, 26, 27, 28, 29, 30, 31},
 'Q3': {4, 5, 6, 7, 12, 13, 14, 15, 20, 21, 22, 23, 28, 29, 30, 31},
 'Q2': {2, 3, 6, 7, 10, 11, 14, 15, 18, 19, 22, 23, 26, 27, 30, 31},
 'Q1': {1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31}}

In [26]:
saving_data(X_test, y_test, test_q_maps, single_qubit_dataset_path, test_file_name, data_type = "Test" )

DRaw_C_Te_v0-002_Q5 created succesfully unique labels: [0 1] 560000
DRaw_C_Te_v0-002_Q4 created succesfully unique labels: [0 1] 560000
DRaw_C_Te_v0-002_Q3 created succesfully unique labels: [0 1] 560000
DRaw_C_Te_v0-002_Q2 created succesfully unique labels: [0 1] 560000
DRaw_C_Te_v0-002_Q1 created succesfully unique labels: [0 1] 560000


In [ ]:
# 000101 5 decimal, qubit label order 5 4 3 2 1